# Hybrid v3 — Anisotropic Alignment + Anisotropic Noise + Speed Variation

**All three mechanisms combined:**

1. **Anisotropic alignment** — neighbors ahead are weighted more heavily in the heading average. This is the key change: the alignment signal itself becomes directional, not just the noise.
2. **Anisotropic noise** — forward neighbors suppress noise (from v1)
3. **Speed variation** — local order controls speed (from v2)

**Why this should produce the forward void:**
When an agent weights forward neighbors more in its alignment, it "locks on" to agents ahead. These agents are being tracked precisely, so they appear at a consistent distance ahead — creating a depleted zone in the body-centered map. Agents behind contribute less to alignment, so the relationship is asymmetric: the follower tracks the leader, but the leader doesn't track the follower as strongly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, os
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

FIGURES_DIR = 'figures/'
FPS = 25
DT  = 1.0 / FPS

FIELD_POL_MEAN   = 0.8196
FIELD_POL_STD    = 0.0615
FIELD_NND_MEDIAN = 3.893
FIELD_NND_MEAN   = 4.473
FIELD_TA_STD     = 0.276

Lx = 99.44857450176137
Ly = 55.93982315724077

print("Ready.")

In [ ]:
def run_hybrid_v3(n_agents, Lx, Ly, v_min, v_max, eta_base, lambda_pull,
                  alpha_aniso, cone_half_angle, r_interaction, r_repulsion,
                  dt, n_steps, seed=42):
    """
    Hybrid v3: anisotropic alignment + anisotropic noise + speed variation.
    
    Three mechanisms:
    1. ANISOTROPIC ALIGNMENT: neighbor headings are weighted by
       w_i = (1 - alpha_aniso) + alpha_aniso * cos(rel_bearing_i / 2)^2
       At alpha_aniso=0 this is isotropic Vicsek. At alpha_aniso=1 neighbors
       directly ahead get weight 1.0, neighbors behind get weight 0.0.
       
    2. ANISOTROPIC NOISE: eta_eff = eta_base * (1 - lambda_pull * f_forward)
       (same as v1/v2)
       
    3. SPEED VARIATION: speed_i = v_min + (v_max - v_min) * local_pol_i
       (same as v2)
    
    Parameters
    ----------
    alpha_aniso : float in [0, 1]
        Alignment anisotropy strength. 0 = isotropic Vicsek. 1 = full front-weighting.
    """
    rng = np.random.default_rng(seed)
    
    pos     = rng.uniform([0, 0], [Lx, Ly], size=(n_agents, 2))
    heading = rng.uniform(-np.pi, np.pi, size=n_agents)
    
    pos_history     = np.empty((n_steps + 1, n_agents, 2))
    heading_history = np.empty((n_steps + 1, n_agents))
    speed_history   = np.empty((n_steps + 1, n_agents))
    pos_history[0]     = pos
    heading_history[0] = heading
    speed_history[0]   = (v_min + v_max) / 2
    
    for step in range(n_steps):
        delta = pos[np.newaxis, :, :] - pos[:, np.newaxis, :]
        delta[:, :, 0] -= Lx * np.round(delta[:, :, 0] / Lx)
        delta[:, :, 1] -= Ly * np.round(delta[:, :, 1] / Ly)
        dist = np.hypot(delta[:, :, 0], delta[:, :, 1])
        
        # --- Relative bearing for anisotropic weighting ---
        bearing_abs = np.arctan2(delta[:, :, 1], delta[:, :, 0])
        rel_bearing = bearing_abs - heading[:, np.newaxis]
        rel_bearing = (rel_bearing + np.pi) % (2 * np.pi) - np.pi
        
        # --- Anisotropic alignment ---
        align_mask = (dist < r_interaction).astype(float)  # includes self on diagonal
        
        # Front-weighting: cos(rel_bearing/2)^2 -> front=1, side=0.5, rear=0
        front_weight = np.cos(rel_bearing / 2) ** 2
        # Blend between isotropic (1.0) and front-weighted
        align_weight = (1.0 - alpha_aniso) + alpha_aniso * front_weight  # [1-alpha, 1]
        # Self-interaction weight = 1 (agent always includes own heading at full weight)
        np.fill_diagonal(align_weight, 1.0)
        
        # Weighted alignment: sum(w_j * exp(i * heading_j)) for j in neighbors
        combined_weight = align_mask * align_weight
        unit_vecs = np.exp(1j * heading)
        neighbour_sum = combined_weight @ unit_vecs
        mean_heading = np.angle(neighbour_sum)
        
        # --- Local polarization for speed variation ---
        n_neighbors = np.maximum(align_mask.sum(axis=1), 1)
        local_pol = np.abs(neighbour_sum) / n_neighbors
        speed = v_min + (v_max - v_min) * local_pol
        
        # --- Forward fraction for anisotropic noise ---
        in_range = (dist < r_interaction) & (dist > 0)
        in_cone  = in_range & (np.abs(rel_bearing) < cone_half_angle)
        n_in_range = in_range.sum(axis=1)
        f_forward = in_cone.sum(axis=1) / np.maximum(n_in_range, 1)
        
        # --- Anisotropic noise ---
        eta_eff = eta_base * (1.0 - lambda_pull * f_forward)
        noise = rng.uniform(-0.5, 0.5, size=n_agents) * eta_eff
        
        new_heading = mean_heading + noise
        
        # --- Short-range repulsion ---
        rep_mask = (dist < r_repulsion) & (dist > 0)
        has_rep = rep_mask.any(axis=1)
        if has_rep.any():
            rep_dx = -(rep_mask * delta[:, :, 0]).sum(axis=1)
            rep_dy = -(rep_mask * delta[:, :, 1]).sum(axis=1)
            rep_heading = np.arctan2(rep_dy, rep_dx)
            new_heading[has_rep] = rep_heading[has_rep] + noise[has_rep]
        
        heading = new_heading
        pos[:, 0] = (pos[:, 0] + speed * dt * np.cos(heading)) % Lx
        pos[:, 1] = (pos[:, 1] + speed * dt * np.sin(heading)) % Ly
        
        pos_history[step + 1] = pos
        heading_history[step + 1] = heading
        speed_history[step + 1] = speed
    
    return pos_history, heading_history, speed_history

print("Hybrid v3 model defined.")

In [ ]:
def polarization_from_headings(hh):
    return np.abs(np.exp(1j * hh).mean(axis=1))

def sim_turning_angles(hh):
    dtheta = np.diff(hh, axis=0)
    return ((dtheta + np.pi) % (2 * np.pi) - np.pi).ravel()

def sim_nnd(ph, Lx, Ly, subsample=20):
    nnds = []
    for t in range(0, len(ph), subsample):
        pos = ph[t]
        d = pos[np.newaxis,:,:] - pos[:,np.newaxis,:]
        d[:,:,0] -= Lx * np.round(d[:,:,0] / Lx)
        d[:,:,1] -= Ly * np.round(d[:,:,1] / Ly)
        dd = np.hypot(d[:,:,0], d[:,:,1])
        np.fill_diagonal(dd, np.inf)
        nnds.extend(np.min(dd, axis=1))
    return np.array(nnds)

def sim_neighbor_density_map(ph, hh, Lx, Ly, radius=10.0, nbins=50, subsample=5):
    edges = np.linspace(-radius, radius, nbins + 1)
    hist = np.zeros((nbins, nbins))
    for t in range(0, len(ph), subsample):
        pos, theta = ph[t], hh[t]
        d = pos[np.newaxis,:,:] - pos[:,np.newaxis,:]
        d[:,:,0] -= Lx * np.round(d[:,:,0] / Lx)
        d[:,:,1] -= Ly * np.round(d[:,:,1] / Ly)
        dd = np.hypot(d[:,:,0], d[:,:,1])
        for i in range(len(pos)):
            m = (dd[i] > 0.1) & (dd[i] < radius)
            if not m.any(): continue
            rel = d[i, m]
            r = np.pi/2 - theta[i]
            rx = rel[:,0]*np.cos(r) - rel[:,1]*np.sin(r)
            ry = rel[:,0]*np.sin(r) + rel[:,1]*np.cos(r)
            h, _, _ = np.histogram2d(rx, ry, bins=edges)
            hist += h
    return hist, edges

print("Metrics defined.")

## Sweeps: first find good alpha_aniso, then 2D sweep eta x lambda

In [ ]:
SHARED = dict(
    n_agents=150, Lx=Lx, Ly=Ly,
    v_min=2.7, v_max=11.76,
    r_interaction=7.0, r_repulsion=1.5,
    dt=DT, cone_half_angle=np.pi / 3,
    lambda_pull=0.2,  # keep from v2
)
BURN_IN = 500

# Step 1: sweep alpha_aniso at fixed eta=1.3 to see its effect
print("=== Sweep alpha_aniso at eta=1.3, lambda=0.2 ===")
alpha_vals = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
for alpha in alpha_vals:
    _, hh, _ = run_hybrid_v3(**SHARED, eta_base=1.3, alpha_aniso=alpha, n_steps=2000, seed=42)
    pol = polarization_from_headings(hh)[BURN_IN:]
    ta = sim_turning_angles(hh[BURN_IN:])
    print(f"  alpha={alpha:.1f}  pol={pol.mean():.3f}  ta_std={np.std(ta):.3f}")

# Step 2: 2D sweep eta_base x lambda at alpha_aniso=0.6 (moderate anisotropy)
print("\n=== 2D sweep eta_base x lambda at alpha=0.6 ===")
eta_vals    = np.arange(0.3, 1.8, 0.1)
lambda_vals = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

pol_grid = np.zeros((len(eta_vals), len(lambda_vals)))
for i, eta in enumerate(eta_vals):
    for j, lam in enumerate(lambda_vals):
        _, hh, _ = run_hybrid_v3(**{**SHARED, 'lambda_pull': lam},
                                  eta_base=eta, alpha_aniso=0.6, n_steps=1500, seed=42)
        pol = polarization_from_headings(hh)[BURN_IN:]
        pol_grid[i, j] = pol.mean()
    print(f"  eta={eta:.1f}  pol = {['%.3f' % p for p in pol_grid[i]]}")

np.save('hybrid_eta_lambda_sweep.npy', pol_grid)
np.save('hybrid_eta_vals.npy', eta_vals)
np.save('hybrid_lambda_vals.npy', lambda_vals)

err = np.abs(pol_grid - FIELD_POL_MEAN)
best_i, best_j = np.unravel_index(err.argmin(), err.shape)
best_eta = eta_vals[best_i]
best_lam = lambda_vals[best_j]
print(f"\nBest: eta={best_eta:.1f}, lambda={best_lam:.1f}  pol={pol_grid[best_i, best_j]:.3f}")

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pol_grid, origin='lower', aspect='auto',
               extent=[lambda_vals[0]-0.1, lambda_vals[-1]+0.1,
                       eta_vals[0]-0.05, eta_vals[-1]+0.05],
               cmap='RdYlBu_r', vmin=0.4, vmax=1.0)
ax.plot(best_lam, best_eta, 'k*', markersize=15, label=f'Best: eta={best_eta:.1f}, lam={best_lam:.1f}')
cs = ax.contour(lambda_vals, eta_vals, pol_grid, levels=[FIELD_POL_MEAN],
                colors='black', linewidths=2, linestyles='--')
ax.clabel(cs, fmt=f'Phi={FIELD_POL_MEAN:.3f}')
ax.set(xlabel='lambda', ylabel='eta_base (rad)',
       title='Hybrid v3: Polarization (alpha_aniso=0.6)')
plt.colorbar(im, ax=ax, label='Polarization')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'hybrid_noise_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Final calibrated run + full metrics + export

In [ ]:
BEST_ALPHA = 0.6  # from alpha sweep above — adjust if sweep suggests different

print(f"Final run: eta={best_eta:.1f}, lambda={best_lam:.1f}, alpha={BEST_ALPHA}")
ph, hh, sh = run_hybrid_v3(
    **{**SHARED, 'lambda_pull': best_lam},
    eta_base=best_eta, alpha_aniso=BEST_ALPHA, n_steps=2000, seed=42
)

pol_sim    = polarization_from_headings(hh)
pol_steady = pol_sim[BURN_IN:]
ta_sim     = sim_turning_angles(hh[BURN_IN:])
speeds     = sh[BURN_IN:].ravel()

print(f"Polarization: {pol_steady.mean():.3f} +/- {pol_steady.std():.3f}  (field: {FIELD_POL_MEAN:.3f})")
print(f"Turning angle std: {np.std(ta_sim):.3f}  (field: {FIELD_TA_STD:.3f})")
print(f"Speed: mean={speeds.mean():.2f} cm/s")

nnd_sim = sim_nnd(ph[BURN_IN:], Lx, Ly)
print(f"NND: median={np.median(nnd_sim):.2f}  mean={nnd_sim.mean():.2f}  (field: {FIELD_NND_MEDIAN:.2f} / {FIELD_NND_MEAN:.2f})")

print("\nComputing neighbor density map...")
hist_sim, edges = sim_neighbor_density_map(ph[BURN_IN:], hh[BURN_IN:], Lx, Ly, radius=10.0, subsample=10)

# --- Neighbor density map ---
fig, ax = plt.subplots(figsize=(7, 6))
hist_norm = hist_sim / (hist_sim.sum() + 1e-10)
im = ax.imshow(hist_norm.T, origin='lower',
               extent=[edges[0], edges[-1], edges[0], edges[-1]],
               cmap='hot', aspect='equal')
ax.plot(0, 0, 'w^', markersize=12)
ax.axhline(0, color='white', ls='--', alpha=0.3)
ax.axvline(0, color='white', ls='--', alpha=0.3)
ax.set(xlabel='Left <- -> Right (cm)', ylabel='Behind <- -> Ahead (cm)',
       title=f'Hybrid v3 — Neighbor density\n(eta={best_eta:.1f}, lam={best_lam:.1f}, alpha={BEST_ALPHA})')
plt.colorbar(im, ax=ax, label='Relative density')
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'hybrid_metric_3.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Comparison table ---
print("\n" + "=" * 75)
print(f"{'Metric':<30} {'Field':>8} {'Vicsek':>8} {'Hyb v1':>8} {'Hyb v2':>8} {'Hyb v3':>8}")
print("=" * 75)
print(f"{'Polarization':<30} {FIELD_POL_MEAN:>8.3f} {'0.824':>8} {'0.812':>8} {'0.818':>8} {pol_steady.mean():>8.3f}")
print(f"{'Turning angle std':<30} {FIELD_TA_STD:>8.3f} {'0.595':>8} {'0.622':>8} {'0.658':>8} {np.std(ta_sim):>8.3f}")
print(f"{'NND median (cm)':<30} {FIELD_NND_MEDIAN:>8.2f} {'2.57':>8} {'2.47':>8} {'2.41':>8} {np.median(nnd_sim):>8.2f}")
print(f"{'NND mean (cm)':<30} {FIELD_NND_MEAN:>8.2f} {'2.93':>8} {'2.75':>8} {'2.66':>8} {nnd_sim.mean():>8.2f}")
print("=" * 75)

# --- Export JSON ---
results = {
    "hybrid": {
        "n_agents": SHARED['n_agents'],
        "Lx": SHARED['Lx'], "Ly": SHARED['Ly'],
        "v_min": SHARED['v_min'], "v_max": SHARED['v_max'],
        "speed": float(speeds.mean()),
        "eta_base": float(best_eta),
        "lambda_pull": float(best_lam),
        "alpha_aniso": float(BEST_ALPHA),
        "cone_half_angle": float(SHARED['cone_half_angle']),
        "eta_empirical": 0.26,
        "eta_calibrated": float(best_eta),
        "r_interaction": SHARED['r_interaction'],
        "r_repulsion": SHARED['r_repulsion'],
        "dt": SHARED['dt'],
        "n_steps": 2000, "burn_in": BURN_IN,
    },
    "metrics": {
        "polarization_mean": float(pol_steady.mean()),
        "polarization_std": float(pol_steady.std()),
        "turning_angle_std": float(np.std(ta_sim)),
        "nnd_median": float(np.median(nnd_sim)),
        "nnd_mean": float(nnd_sim.mean()),
    },
    "field_targets": {
        "polarization_mean": FIELD_POL_MEAN, "polarization_std": FIELD_POL_STD,
        "turning_angle_std": FIELD_TA_STD,
        "nnd_median": FIELD_NND_MEDIAN, "nnd_mean": FIELD_NND_MEAN,
    }
}

with open('week2_hybrid_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved: week2_hybrid_results.json")

print("\n--- Output verification ---")
for f in ['week2_hybrid_results.json', 'hybrid_eta_lambda_sweep.npy',
          'hybrid_eta_vals.npy', 'hybrid_lambda_vals.npy',
          'figures/hybrid_metric_3.png', 'figures/hybrid_noise_sweep.png']:
    print(f"  {f}: {'OK' if os.path.exists(f) else 'MISSING'}")